# 🚁 University-1652 Cross-View Geo-Localization
## RGB & RGBD Training/Testing Pipeline

Bu notebook, University-1652 dataset üzerinde:
- **RGB Baseline** eğitimi (satellite + drone, 2-view)
- **RGBD** eğitimi (satellite + depth, 4-channel)
- **LPN** (Local Pattern Network) desteği
- **Circle Loss** ve diğer metric learning loss'ları
- **Test & Evaluation** (Drone→Satellite, Satellite→Drone)

adımlarını içerir.

---

## 1️⃣ GPU Kontrolü & Runtime Ayarı
> **Runtime → Change runtime type → T4 GPU** seçtiğinizden emin olun!

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ GPU bulunamadı! Runtime → Change runtime type → T4 GPU")

## 2️⃣ Google Drive Bağlama

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3️⃣ Repo Klonlama & Requirements Kurulumu

In [ ]:
import os
os.chdir('/content')

# Repo'yu klonla (GitHub kullanıcı adınızı değiştirin)
!git clone https://github.com/KULLANICI_ADINIZ/University1652-Fork.git
os.chdir('/content/University1652-Fork')
print(f"\n✅ Working directory: {os.getcwd()}")

In [ ]:
# Requirements kurulumu
!pip install -q matplotlib pyyaml
!pip install -q pytorch_metric_learning
!pip install -q timm

# Doğrulama
import pytorch_metric_learning
print(f"✅ pytorch_metric_learning: {pytorch_metric_learning.__version__}")
print(f"✅ torch: {torch.__version__}")

## 4️⃣ Dataset Hazırlama

University-1652 dataset'ini Google Drive'a yükleyin ve aşağıdaki path'i güncelleyin.

**Beklenen yapı:**
```
University-1652/
├── train/
│   ├── satellite/    (701 class folders)
│   ├── drone/        (701 class folders)
│   └── street/       (701 class folders, opsiyonel)
└── test/
    ├── query_satellite/
    ├── query_drone/
    ├── gallery_satellite/
    └── gallery_drone/
```

In [ ]:
# ============================================================
# ⚙️ AYARLAR — Bu hücredeki path'leri kendi yapınıza göre değiştirin
# ============================================================

# Dataset zip dosyasının Drive'daki yolu
DATASET_ZIP = '/content/drive/MyDrive/University-1652.zip'

# Dataset'in açılacağı yer
DATASET_ROOT = '/content/University-1652'

# Train & Test dizinleri (zip açıldıktan sonra)
TRAIN_DIR = os.path.join(DATASET_ROOT, 'train')
TEST_DIR  = os.path.join(DATASET_ROOT, 'test')

In [ ]:
# Dataset'i Drive'dan aç
if os.path.exists(DATASET_ZIP):
    print(f"📦 Dataset açılıyor: {DATASET_ZIP}")
    !unzip -q -n {DATASET_ZIP} -d /content/
    print("✅ Dataset açıldı!")
elif os.path.exists(DATASET_ROOT):
    print(f"✅ Dataset zaten mevcut: {DATASET_ROOT}")
else:
    print(f"❌ Dataset bulunamadı!")
    print(f"   Beklenen zip: {DATASET_ZIP}")
    print(f"   Veya klasör:  {DATASET_ROOT}")
    print("\n💡 Dataset'i Drive'a yükleyip path'i güncelleyin.")

In [ ]:
# Dataset yapısını doğrula
for split, split_dir in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    print(f"\n📂 {split}: {split_dir}")
    if os.path.exists(split_dir):
        for subdir in sorted(os.listdir(split_dir)):
            subdir_path = os.path.join(split_dir, subdir)
            if os.path.isdir(subdir_path):
                n = len([d for d in os.listdir(subdir_path) if os.path.isdir(os.path.join(subdir_path, d))])
                print(f"   └── {subdir}: {n} classes")
    else:
        print(f"   ❌ Dizin bulunamadı!")

---
## 5️⃣ RGB Baseline Eğitimi

Standart 3-kanallı RGB satellite + drone eğitimi.

In [ ]:
os.chdir('/content/University1652-Fork')

# =============================================
# RGB Baseline Training (2-view: Satellite + Drone)
# =============================================
!python train.py \
    --name rgb_baseline \
    --data_dir {TRAIN_DIR} \
    --gpu_ids 0 \
    --batchsize 8 \
    --h 256 --w 256 \
    --views 2 \
    --pool avg \
    --lr 0.01 \
    --droprate 0.5 \
    --stride 2 \
    --erasing_p 0.5 \
    --color_jitter \
    --fp16

### 5B: RGB + Circle Loss Eğitimi

In [ ]:
# =============================================
# RGB + Circle Loss Training
# =============================================
!python train.py \
    --name rgb_circle \
    --data_dir {TRAIN_DIR} \
    --gpu_ids 0 \
    --batchsize 8 \
    --h 256 --w 256 \
    --views 2 \
    --pool avg \
    --lr 0.01 \
    --droprate 0.5 \
    --stride 2 \
    --circle \
    --fp16

### 5C: RGB + LPN Eğitimi

In [ ]:
# =============================================
# RGB + LPN Training (Local Pattern Network)
# =============================================
!python train.py \
    --name rgb_lpn \
    --data_dir {TRAIN_DIR} \
    --gpu_ids 0 \
    --batchsize 8 \
    --h 256 --w 256 \
    --views 2 \
    --pool lpn \
    --lpn_blocks 4 \
    --lpn_mode square \
    --lr 0.01 \
    --droprate 0.5 \
    --stride 2 \
    --fp16

---
## 6️⃣ RGBD Eğitimi (Depth Map Gerekli)

### 6A: MiDaS ile Satellite Depth Map Oluşturma

In [ ]:
# =============================================
# MiDaS Depth Map Generation (satellite images)
# =============================================
import torch
import cv2
import numpy as np
from tqdm import tqdm

# MiDaS yükle
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.small_transform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)
midas.eval()
print(f"✅ MiDaS yüklendi ({device})")

In [ ]:
def generate_depth_maps(satellite_dir, depth_dir):
    """Satellite görüntüleri için depth map oluştur"""
    os.makedirs(depth_dir, exist_ok=True)
    
    class_folders = sorted([d for d in os.listdir(satellite_dir) 
                           if os.path.isdir(os.path.join(satellite_dir, d))])
    print(f"📊 {len(class_folders)} class folder işlenecek")
    
    total_processed = 0
    total_skipped = 0
    
    for cls_folder in tqdm(class_folders, desc="Depth maps"):
        cls_path = os.path.join(satellite_dir, cls_folder)
        depth_cls_path = os.path.join(depth_dir, cls_folder)
        os.makedirs(depth_cls_path, exist_ok=True)
        
        for img_name in os.listdir(cls_path):
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            if img_name.startswith('._'):
                continue
            
            depth_name = os.path.splitext(img_name)[0] + '_depth.jpg'
            depth_path = os.path.join(depth_cls_path, depth_name)
            
            if os.path.exists(depth_path):
                total_skipped += 1
                continue
            
            img_path = os.path.join(cls_path, img_name)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            input_batch = transform(img_rgb).to(device)
            
            with torch.no_grad():
                prediction = midas(input_batch)
                prediction = torch.nn.functional.interpolate(
                    prediction.unsqueeze(1),
                    size=img.shape[:2],
                    mode="bicubic",
                    align_corners=False,
                ).squeeze()
            
            depth = prediction.cpu().numpy()
            depth_norm = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)
            cv2.imwrite(depth_path, depth_norm.astype(np.uint8))
            total_processed += 1
    
    print(f"\n✅ Tamamlandı! İşlenen: {total_processed}, Atlanan: {total_skipped}")

# Train satellite depth map'leri
print("=" * 50)
print("Train Satellite Depth Maps")
print("=" * 50)
generate_depth_maps(
    satellite_dir=os.path.join(TRAIN_DIR, 'satellite'),
    depth_dir=os.path.join(TRAIN_DIR, 'satellite_depth')
)

In [ ]:
# Test satellite depth map'leri (query_satellite için)
print("=" * 50)
print("Test Query Satellite Depth Maps")
print("=" * 50)

query_sat_dir = os.path.join(TEST_DIR, 'query_satellite')
if os.path.exists(query_sat_dir):
    generate_depth_maps(
        satellite_dir=query_sat_dir,
        depth_dir=os.path.join(TEST_DIR, 'query_satellite_depth')
    )
else:
    print(f"⚠️ query_satellite klasörü bulunamadı: {query_sat_dir}")

### 6B: RGBD Model Eğitimi

In [ ]:
os.chdir('/content/University1652-Fork')

# =============================================
# RGBD Training (4-channel satellite input)
# =============================================
!python train.py \
    --name rgbd_baseline \
    --data_dir {TRAIN_DIR} \
    --use_rgbd \
    --gpu_ids 0 \
    --batchsize 8 \
    --h 256 --w 256 \
    --views 2 \
    --pool avg \
    --lr 0.01 \
    --droprate 0.5 \
    --stride 2 \
    --erasing_p 0.5 \
    --color_jitter \
    --fp16

### 6C: RGBD + LPN Eğitimi

In [ ]:
# =============================================
# RGBD + LPN Training
# =============================================
!python train.py \
    --name rgbd_lpn \
    --data_dir {TRAIN_DIR} \
    --use_rgbd \
    --gpu_ids 0 \
    --batchsize 8 \
    --h 256 --w 256 \
    --views 2 \
    --pool lpn \
    --lpn_blocks 4 \
    --lpn_mode square \
    --lr 0.01 \
    --droprate 0.5 \
    --stride 2 \
    --fp16

### 6D: Depth klasörü ayrı bir yerdeyse (--depth_dir ile)

In [ ]:
# =============================================
# RGBD Training — ayrı depth dizini kullanarak
# depth_dir: satellite_depth klasörünün PARENT dizini
# =============================================

# Örneğin depth dosyaları /content/depth_data/satellite_depth/ altındaysa:
# DEPTH_DIR = '/content/depth_data'

# !python train.py \
#     --name rgbd_separate_depth \
#     --data_dir {TRAIN_DIR} \
#     --use_rgbd \
#     --depth_dir /content/depth_data \
#     --gpu_ids 0 \
#     --batchsize 8 \
#     --h 256 --w 256 \
#     --views 2 \
#     --pool avg \
#     --lr 0.01 \
#     --droprate 0.5 \
#     --fp16

---
## 7️⃣ Test & Evaluation

### 7A: RGB Baseline Test (Drone → Satellite)

In [ ]:
os.chdir('/content/University1652-Fork')

# =============================================
# Test: Drone → Satellite (varsayılan yön)
# =============================================
MODEL_NAME = 'rgb_baseline'  # Test etmek istediğiniz modelin adı

!python test.py \
    --name {MODEL_NAME} \
    --test_dir {TEST_DIR} \
    --gpu_ids 0 \
    --h 256 --w 256 \
    --views 2

In [ ]:
# Sonuçları göster
!cat ./model/{MODEL_NAME}/result.txt

### 7B: Ters Yön Test (Satellite → Drone)

In [ ]:
# =============================================
# Test: Satellite → Drone (ters yön)
# =============================================
!python test.py \
    --name {MODEL_NAME} \
    --test_dir {TEST_DIR} \
    --gpu_ids 0 \
    --h 256 --w 256 \
    --views 2 \
    --query_folder query_drone \
    --gallery_folder gallery_satellite

### 7C: RGBD Model Test

In [ ]:
# =============================================
# RGBD Model Test
# test.py opts.yaml'dan use_rgbd değerini otomatik okur
# =============================================
RGBD_MODEL_NAME = 'rgbd_baseline'

!python test.py \
    --name {RGBD_MODEL_NAME} \
    --test_dir {TEST_DIR} \
    --gpu_ids 0 \
    --h 256 --w 256 \
    --views 2

In [ ]:
# RGBD sonuçları
!cat ./model/{RGBD_MODEL_NAME}/result.txt

### 7D: Retrieval Görselleştirme (Top-10)

In [ ]:
# Önce test yapılmalı, sonra demo çalıştırılabilir
!python demo.py --query_index 0

from IPython.display import Image, display
if os.path.exists('show.png'):
    display(Image('show.png', width=800))
else:
    print("⚠️ show.png bulunamadı. Önce test.py çalıştırın.")

---
## 8️⃣ Modeli Google Drive'a Kaydetme

In [ ]:
import shutil

SAVE_MODEL_NAME = 'rgb_baseline'  # Kaydetmek istediğiniz model

src = f'./model/{SAVE_MODEL_NAME}'
dst = f'/content/drive/MyDrive/University1652_models/{SAVE_MODEL_NAME}'

if os.path.exists(src):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"✅ Model kaydedildi: {dst}")
    # Dosya listesi
    for f in os.listdir(dst):
        size = os.path.getsize(os.path.join(dst, f)) / 1e6
        print(f"   {f}: {size:.1f} MB")
else:
    print(f"❌ Model bulunamadı: {src}")

## 9️⃣ Eğitimi Kaldığı Yerden Devam Ettirme (Resume)

In [ ]:
# Drive'dan modeli geri yükle (Colab bağlantısı kopmuşsa)
import shutil

RESUME_MODEL_NAME = 'rgb_baseline'

src = f'/content/drive/MyDrive/University1652_models/{RESUME_MODEL_NAME}'
dst = f'./model/{RESUME_MODEL_NAME}'

os.makedirs('./model', exist_ok=True)
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copytree(src, dst)
    print(f"✅ Model geri yüklendi: {dst}")
elif os.path.exists(dst):
    print(f"✅ Model zaten mevcut: {dst}")
else:
    print(f"❌ Drive'da model bulunamadı: {src}")

In [ ]:
# Resume ile devam et
!python train.py \
    --name {RESUME_MODEL_NAME} \
    --resume \
    --gpu_ids 0 \
    --fp16

---
## 📊 Parametre Referans Tablosu

| Parametre | Açıklama | Önerilen Değer |
|-----------|----------|----------------|
| `--name` | Deney adı (model/ altına kaydedilir) | `rgb_baseline` |
| `--data_dir` | Train veri dizini | `/content/University-1652/train` |
| `--views` | View sayısı (2: sat+drone, 3: +street) | `2` |
| `--pool` | Pooling: avg, max, gem, avg+max, **lpn** | `avg` |
| `--lpn_blocks` | LPN blok sayısı (square: tam kare olmalı) | `4` |
| `--lpn_mode` | LPN modu: square veya horizontal | `square` |
| `--use_rgbd` | RGBD (4 kanal) satellite input | flag |
| `--depth_dir` | Ayrı depth dizini (yoksa data_dir kullanılır) | `None` |
| `--circle` | Circle Loss ekle | flag |
| `--triplet` | Triplet Loss ekle | flag |
| `--fp16` | Half precision (bellek tasarrufu) | flag |
| `--batchsize` | Batch boyutu (GPU'ya göre) | `8` (T4), `16` (A100) |
| `--h --w` | Giriş boyutu | `256` (hızlı), `384` (kaliteli) |
| `--stride` | Son layer stride | `2` (normal), `1` (daha iyi, daha yavaş) |